# DEAP (Distributed Evolutionary Algorithms in Python)

### https://deap.readthedocs.io/

## ``Creator Module``
    - Help create classes/types for EA

## ``Toolbox Module``
    - Create a toolbox to contain EA operators

## ``01. Setup``

`base` = framework skeleton

``creator`` = define problem-specific objects

``tools`` = ready-made operators

In [2]:
import random
from deap import base  
# Provides the base framework to register and manage functions (e.g., evaluation, selection, mutation).

from deap import creator  
# Used to define custom classes (e.g., Fitness, Individual) with attributes needed for the algorithm.

from deap import tools  
# Provides ready-to-use operators like selection, crossover, mutation, and statistics helpers.

import matplotlib.pyplot as plt


## ``02. Problem Constants``

In [ ]:
# problem constants:
ONE_MAX_LENGTH = 100   # Each individual (chromosome) is a list of 100 bits (0s and 1s)

# Genetic Algorithm constants:
POPULATION_SIZE = 200  # There are 200 individuals (chromosomes) in the population
P_CROSSOVER = 0.9      # 90% chance that crossover (recombination) happens between parents
P_MUTATION = 0.1       # 10% chance that mutation flips a bit in an individual
MAX_GENERATIONS = 100  # The algorithm will evolve the population for 100 generations


## ``03. Setup Toolbox with Creator``

`Why set a random seed?`
In evolutionary algorithms, randomness is everywhere (initial population, mutation, crossover, selection).
Without a seed → every run will give different results.
With a fixed seed (like 42) → the random sequence is reproducible, so you can debug and compare results reliably.

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

None


-----------------------------------------------------------------------

In [11]:
from deap import base, creator

# Step 1: Create the class
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Step 2: Use it to make a fitness object
fit = creator.FitnessMax()

# Step 3: Assign a fitness value
fit.values = (10,)   # means "this solution has fitness score = 10"

print(fit.values)    # (10,)
print(type(fit))     # <class 'deap.creator.FitnessMax'>

(10.0,)
<class 'deap.creator.FitnessMax'>


----------------------------------------------

Creates a new fitness class for single-objective maximization.

We need this because DEAP must know whether to maximize or minimize the fitness.

weights=(1.0,) means we only have one objective, and higher values are better.

In [19]:
# Define a fitness class for single-objective maximization (higher fitness values are better).
creator.create("FitnessMax", base.Fitness, weights=(1.0,)) 


In [20]:
# Create individual class (list with a FitnessMax attribute)
creator.create("Individual", list, fitness=creator.FitnessMax)

----------------------------------

In [17]:
from deap import base, creator

# Step 1: Create fitness class (maximize single objective)
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Step 2: Create individual class (list with a FitnessMax attribute)
creator.create("Individual", list, fitness=creator.FitnessMax)

# Step 3: Make an individual
ind = creator.Individual([0, 1, 1, 0, 1])

print(ind)              # [0, 1, 1, 0, 1]  (behaves like a list)
print(ind.fitness)      # <deap.creator.FitnessMax object at ...>


[0, 1, 1, 0, 1]
()


In [18]:
ind.fitness.values = (5,)
print(ind.fitness.values)   # (5,)

(5.0,)


--------------------------------------

### `Register Operators`

In [23]:
toolbox = base.Toolbox()

# • Toolbox is like a registry of functions.
# • You “register” building blocks (random generators, individual creators, population creators, operators) into it.
# • Later, instead of writing long function calls, you just call toolbox.something().

# Think of it as your recipe book for building and evolving individuals.

In [24]:
toolbox.register("zeroOrOne", random.randint, 0, 1)

# 	• This registers a function called "zeroOrOne".
# 	• It uses random.randint(0,1) → returns either 0 or 1.
# That’s how each gene (bit) in a chromosome will be created.

In [ ]:
toolbox.register("individualCreator", 
                 tools.initRepeat, creator.Individual, 
                 toolbox.zeroOrOne, ONE_MAX_LENGTH)

# 	• tools.initRepeat → a DEAP helper that repeats a function N times and stores the results.
# 	• creator.Individual → the class you defined earlier (a list with a fitness attached).
# 	• toolbox.zeroOrOne → the function that generates a random 0 or 1.
# ONE_MAX_LENGTH → how many times to repeat it (the chromosome length).

# So this creates an individual like:

# [1, 0, 1, 1, 0, 0, 1, ...]   # length = ONE_MAX_LENGTH


# and wraps it as a creator.Individual (so it has a .fitness attribute).

In [26]:
toolbox.register("populationCreator", 
                 tools.initRepeat, list, 
                 toolbox.individualCreator)

# Again using tools.initRepeat.

# list → the base container for the population.

# toolbox.individualCreator → function to generate one individual.

# By default, this will repeat until you ask it how many individuals you want (e.g., POPULATION_SIZE).

--------------------------------------------------------------

In [27]:
pop = toolbox.populationCreator(n=5)  # make 5 individuals
for ind in pop:
    print(ind, ind.fitness.values)


[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1] ()
[1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0] ()
[0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0] ()
[0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 

## Till Now 

- You defined FitnessMax → tells DEAP “maximize fitness.”

- You defined Individual → a list with that fitness.

- Now with the toolbox, you define how to actually build those individuals and populations.

Without the toolbox, you’d have to manually write loops to create random chromosomes. With it, everything is standardized.

------------------------

## `Register Genetic Operators`

`Selection`

In [ ]:
toolbox.register("select", tools.selRoulette)

# Purpose: Choose individuals from the current population to be parents for the next generation.

# tools.selRoulette → roulette wheel selection: probability of being selected is proportional to fitness.

# Higher fitness → higher chance of being picked.

# parents = toolbox.select(pop, k=2)
# Picks 2 individuals from pop based on fitness.

`Crossover`

In [ ]:
toolbox.register("mate", tools.cxOnePoint)

# Purpose: Combine two parents to produce offspring (next generation).
# one-point crossover:

    # Pick a random point in the chromosome.

    # Swap the tails of the two parents to create two children.

# Parameters:

# When called, it expects two individuals: toolbox.mate(parent1, parent2).

# child1, child2 = toolbox.mate(parent1, parent2)
# If parents are [0,1,1,0] and [1,0,0,1] and crossover point = 2:

# Child1 = [0,1,0,1]

# Child2 = [1,0,1,0]

In [ ]:


toolbox.register("mutate", tools.mutFlipBit,
indpb=1.0/ONE_MAX_LENGTH)

toolbox.register("select", tools.selRoulette)
toolbox.register("mate", tools.cxOnePoint)
toolbox.register("mutate", tools.mutFlipBit,
indpb=1.0/ONE_MAX_LENGTH)